# Exercises (Student) - MCP Client with LLM

In [7]:
!pip install -q mcp nest_asyncio requests

In [ ]:

import os
from pathlib import Path
MCP_HTTP_TOKEN = os.getenv("MCP_HTTP_TOKEN", "devtoken123")
USE_REAL_LLM = False  # flip True if GITHUB_TOKEN is set


In [ ]:
import asyncio
import nest_asyncio
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

nest_asyncio.apply()


In [31]:
%%writefile server.py
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("DemoServer")

@mcp.tool()
def add(a: int, b: int) -> int:
    "Add two numbers."
    return a + b

# TODO (optional exercise): add multiply(a, b) here if you want an extra tool

@mcp.tool()
def greet(name: str) -> str:
    "Return a greeting string."
    return f"Hello, {name}!"

if __name__ == "__main__":
    mcp.run()

Overwriting server.py


## Exercise 1 (provide answer)

#To-Do: Why is STDIO transport simple for local MCP dev compared to HTTP?

## Exercise 2

In [11]:
import asyncio
import nest_asyncio
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

nest_asyncio.apply()

async def ex2_connect():
    params = StdioServerParameters(command="python", args=["server.py"])
    async with stdio_client(params, errlog=True) as (r, w):
        async with ClientSession(r, w) as session:
            await session.initialize()

## Exercise 3

In [25]:
async def ex3_list():
    params = StdioServerParameters(command="python", args=["server.py"])
    async with stdio_client(params, errlog=True) as (r, w):
        async with ClientSession(r, w) as session:
            await session.initialize()
            resources = await session.list_resources()
            print("RESOURCES:", resources)
            tools = await session.list_tools()
            for t in tools.tools:
                print(t.name, t.inputSchema.get("properties", {}))

## Exercise 4

#To-Do: Explain how the conversion to llm tool happens in MCP server code ?

In [26]:

def convert_to_llm_tool(tool):
    return {
        "type": "function",
        "function": {
            "name": tool.name,
            "description": tool.description or "mcp tool",
            "parameters": {
                "type": "object",
                "properties": tool.inputSchema.get("properties", {}),
                "required": tool.inputSchema.get("required", []),
            },
        },
    }


## Exercise 5

**Plan & execute:** Use stub (or real) LLM to propose `tool_calls`, then execute them and print results for a prompt like “Add 2 to 20.”

In [27]:

import asyncio
import json
import nest_asyncio
from typing import Any, Dict, List
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

nest_asyncio.apply()

def call_llm(prompt: str, functions: List[Dict[str, Any]], use_real: bool = False):
    if not use_real:
        return stub_plan(prompt, functions)
    token = os.getenv("GITHUB_TOKEN")
    if not token:
        raise RuntimeError("Set GITHUB_TOKEN or use stub planner.")
    from azure.ai.inference import ChatCompletionsClient
    from azure.core.credentials import AzureKeyCredential
    client = ChatCompletionsClient("https://models.inference.ai.azure.com", AzureKeyCredential(token))
    resp = client.complete(
        model="gpt-4o",
        messages=[{"role": "system", "content": "Plan MCP tool calls."},{"role": "user", "content": prompt}],
        tools=functions,
        temperature=0,
        max_tokens=400,
    )
    calls = []
    msg = resp.choices[0].message
    for tc in msg.tool_calls or []:
        args = tc.function.arguments
        args_json = json.loads(args) if isinstance(args, str) else args
        calls.append({"name": tc.function.name, "args": args_json})
    return calls


In [29]:
async def ex5_run(prompt: str = "Add 2 to 20"):
    params = StdioServerParameters(command="python", args=["server.py"])
    async with stdio_client(params, errlog=True) as (r, w):
        async with ClientSession(r, w) as session:
            await session.initialize()
            tools = await session.list_tools()
            functions = [convert_to_llm_tool(t) for t in tools.tools]
            calls = call_llm(prompt, functions, USE_REAL_LLM)
            print("tool_calls:", calls)
            for call in calls:
                result = await session.call_tool(call["name"], arguments=call["args"])
                print("result:", [getattr(c, "text", str(c)) for c in result.content])

## Optional - add multiply(a, b) and rerun